# Phase 30: LSTM Architecture (Deep Learning)

**Goal:** We have exhausted classical ML. Now we step into Deep Learning. We will implement a **Bidirectional Long Short-Term Memory (BiLSTM)** neural network from scratch using PyTorch Lightning.

Unlike Tabular ML (which looks at one event in isolation), the BiLSTM reads sequences of 10 network events forward *and* backward to mathematically understand the temporal flow of an attack!

In [1]:
import sys
!{sys.executable} -m pip install torch pytorch-lightning mlflow numpy pytest scikit-learn  # type: ignore  # pylint: disable=import-error

import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
import numpy as np
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
import os
import mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/852.4 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━ 524.3/852.4 kB 5.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 6.1 MB/s  0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/983.4 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 11.2 MB/s  0:00:00


...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 4/6 [torchmetrics]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 5/6 [pytorch-lightning]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 5/6 [pytorch-lightning]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [pytorch-lightning]


2026/09/07 15:45:39 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Volumes/Fullstack/Github/XAI-Guard/ml/.venv/lib/python3.14/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


### Step 1: BiLSTM Architecture (Subphase 30.1)
We construct the Neural Network architecture. 
- An **Input Projection Layer** to compress raw features.
- The **BiLSTM Engine** which reads the sequence `[batch, seq_len=10, features]`.
- A **Classification Head** with Dropout to prevent overfitting.
- The loss function is strictly weighted `CrossEntropyLoss` with a `clip_grad_norm_` of 1.0 to prevent Exploding Gradients!

In [2]:
class BiLSTMModel(pl.LightningModule):
    def __init__(self, n_features, n_classes, hidden_dim=128, num_layers=2, dropout=0.3, class_weights=None, learning_rate=1e-3):
        super().__init__()
        self.save_hyperparameters()
        
        # 1. Input Projection
        self.projection = nn.Linear(n_features, hidden_dim)
        self.proj_gelu = nn.GELU()
        
        # 2. BiLSTM Engine (Bidirectional = True)
        self.lstm = nn.LSTM(
            input_size=hidden_dim, 
            hidden_size=hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=dropout if num_layers > 1 else 0.0, 
            bidirectional=True
        )
        
        # 3. Classification Head
        # Because it is bidirectional, the output hidden state is 2 * hidden_dim
        self.head = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes)
        )
        
        self.learning_rate = learning_rate
        weight_tensor = torch.tensor(class_weights, dtype=torch.float32) if class_weights is not None else None
        self.criterion = nn.CrossEntropyLoss(weight=weight_tensor)
        self._is_fitted = False
        
    def forward(self, x):
        # x shape: (batch_size, seq_len=10, n_features)
        x = self.proj_gelu(self.projection(x))
        
        # lstm_out shape: (batch_size, seq_len, 2*hidden_dim)
        lstm_out, (h_n, c_n) = self.lstm(x)
        
        # We only care about the LAST time step's output to classify the whole window
        final_timestep = lstm_out[:, -1, :]
        
        logits = self.head(final_timestep)
        return logits

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=1e-4)
        
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        self.log("train_loss", loss)
        return loss
        
    def predict_proba(self, x):
        self.eval()
        with torch.no_grad():
            logits = self(x)
            return F.softmax(logits, dim=-1).cpu().numpy()
            
    # We must enforce Gradient Clipping manually if not using PyTorch Lightning's native trainer wrapper
    # (But Lightning Trainer will actually handle it for us in Step 2!)

print("✅ BiLSTM PyTorch Architecture Registered!")

✅ BiLSTM PyTorch Architecture Registered!


### Step 2: PyTorch Lightning Trainer Config (Subphase 30.2)
We configure the mathematical guardrails for the neural network training loop. 
- `EarlyStopping` prevents overfitting.
- `ModelCheckpoint` saves the exact epoch with the highest Validation F1 Score.
- `gradient_clip_val=1.0` violently forces the mathematics to stay stable by clipping exploding gradients.

In [3]:
def build_trainer_config(experiment_name="lstm_experiment"):
    checkpoint_callback = ModelCheckpoint(
        monitor="val_f1_macro",
        mode="max",
        save_top_k=1,
        dirpath="checkpoints/",
        filename="bilstm-{epoch:02d}-{val_f1_macro:.4f}"
    )
    
    early_stopping = EarlyStopping(
        monitor="val_f1_macro",
        mode="max",
        patience=10
    )
    
    # Detect Apple Silicon (MPS), CUDA (GPU), or fallback to CPU
    if torch.backends.mps.is_available():
        accelerator = "mps"
    elif torch.cuda.is_available():
        accelerator = "gpu"
    else:
        accelerator = "cpu"
        
    trainer = pl.Trainer(
        max_epochs=100,
        accelerator=accelerator,
        callbacks=[checkpoint_callback, early_stopping],
        gradient_clip_val=1.0, # CRITICAL: Gradient Clipping!
        logger=False, # We use native MLflow below
        enable_progress_bar=False
    )
    return trainer

print("✅ PyTorch Lightning Trainer Configured with Gradient Clipping and Checkpointing!")

✅ PyTorch Lightning Trainer Configured with Gradient Clipping and Checkpointing!


### Step 3: Deep Learning Unit Tests (Subphase 30.3)
Deep learning fails silently. Before we burn hours of GPU time training, we strictly test the mathematical properties of our architecture!

In [4]:
print("=== RUNNING DEEP LEARNING ARCHITECTURE TESTS ===")

test_model = BiLSTMModel(n_features=50, n_classes=7, hidden_dim=64, num_layers=1)
dummy_x = torch.randn(32, 10, 50) # Batch Size 32, Sequence Length 10, Features 50
dummy_y = torch.randint(0, 7, (32,))

# Test 1: Forward Pass Output Shape
logits = test_model(dummy_x)
assert logits.shape == (32, 7), f"❌ FAILED Test 1: Shape is {logits.shape}"
print("✅ PASSED Test 1: Forward pass produces correct [batch, classes] shape.")

# Test 2: Predict Proba sums to 1.0
probas = test_model.predict_proba(dummy_x)
assert np.allclose(np.sum(probas, axis=1), 1.0, atol=1e-5), "❌ FAILED Test 2"
print("✅ PASSED Test 2: Softmax mathematically forces probabilities to sum to 1.0.")

# Test 3: Gradient Flow Check
loss = test_model.criterion(logits, dummy_y)
loss.backward()
has_gradients = all(p.grad is not None for p in test_model.parameters() if p.requires_grad)
assert has_gradients, "❌ FAILED Test 3: Some layers are mathematically detached!"
print("✅ PASSED Test 3: Backpropagation gradient flow reaches all layers flawlessly.")

# Test 4: Bidirectional Doubling check
assert test_model.lstm.bidirectional == True
assert test_model.head[0].in_features == 128 # 64 * 2 (Because forward hidden + backward hidden)
print("✅ PASSED Test 4: Bidirectional mechanics correctly double the hidden state.")

print("\n🎯 ALL NEURAL NETWORK ARCHITECTURE TESTS PASSED!")

=== RUNNING DEEP LEARNING ARCHITECTURE TESTS ===


✅ PASSED Test 1: Forward pass produces correct [batch, classes] shape.
✅ PASSED Test 2: Softmax mathematically forces probabilities to sum to 1.0.
✅ PASSED Test 3: Backpropagation gradient flow reaches all layers flawlessly.
✅ PASSED Test 4: Bidirectional mechanics correctly double the hidden state.

🎯 ALL NEURAL NETWORK ARCHITECTURE TESTS PASSED!
